# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Loaded:\n")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Identify all available record sets and their respective field IDs in the dataset. All entities are referenced by their `@id` fields.

In [ ]:
# The mlcroissant API exposes available record sets through the Dataset object's record_sets property.
print("Available Record Sets (by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  - {rs['@id']} (name: {rs.get('name', '<no name>')})")

# Display fields/columns (by @id) for each record set
for rs in record_sets:
    print(f"\nFields/columns for RecordSet '@id': {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        # in schema, could be dict or list
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            print(f"    - {f['@id']} (name: {f.get('name', '<no name>')}, dtype: {f.get('dataType', '<no type>')})")
    elif 'column' in rs:
        columns = rs['column']
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            print(f"    - {col['@id']} (name: {col.get('name', '<no name>')}, dtype: {col.get('dataType', '<no type>')})")
    else:
        print("    - <No fields/columns found>")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Reference each entity by its `@id` field.

In [ ]:
# Get record set @ids as a list
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  -> Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("  -> No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

# Choose the first non-empty dataframe for demonstration
for rid, df in dataframes.items():
    if len(df) > 0:
        selected_record_set_id = rid
        break

print(f"\nSample columns from record set '@id': {selected_record_set_id}")
print(dataframes[selected_record_set_id].columns.tolist())

dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Perform key data processing operations:
- Filter records based on a numeric field (using its `@id` as column label),
- Normalize a chosen numeric field,
- Group by a categorical field (referenced by its `@id`).

In [ ]:
# Choose a numeric and a categorical field @id for EDA
df = dataframes[selected_record_set_id]

# Attempt to auto-detect a numeric field (int/float columns)
import numpy as np

numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if not numeric_field_id:
    # fallback: attempt to convert first column to numeric
    for col in df.columns:
        try:
            s = pd.to_numeric(df[col])
            numeric_field_id = col
            df[col] = s
            break
        except Exception:
            continue

print(f"Using '{numeric_field_id}' as numeric field for filtering and normalization.")

# Select a threshold for filtering (arbitrarily, 10)
threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}.")

    # Normalize the field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print("\nSample of normalized numeric field:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No usable numeric fields found for EDA.")

# Attempt to group by a categorical field
group_field_id = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id and numeric_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships based on the exploratory processing above.

In [ ]:
# Basic visualizations: Histogram and grouped bar chart
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and len(filtered_df) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (filtered > {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id and numeric_field_id and len(filtered_df) > 0:
    plt.figure(figsize=(8,5))
    sns.barplot(y=group_field_id, x=numeric_field_id, data=filtered_df, ci=None)
    plt.title(f"Filtered '{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel(group_field_id)
    plt.show()
else:
    print("Insufficient data for plotting group field.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the FAIR^2 colorectal cancer survivor dataset using `mlcroissant`:

- **Schema-driven loading** ensures robust and reproducible data access using Croissant `@id` references.
- We reviewed available `recordSet` and `field`/`column` identifiers, extracted tabular data as DataFrames, and performed basic EDA (filtering, normalization, grouping).
- Visualizations highlighted sample numeric distributions and groupings.

This workflow may be extended for clinical analysis and further FAIR dataset exploration.